# Phase 3 item 1 (data prep) — vector_db chunking proof

Chunks `context/docs/index.html` for `staging.doc_chunks` → `vector_db.chunks_docs`
(`docs/data-pipeline.md` Part 3). Verifies three things against the *real*
installed `unstructured` and the *real* file before this becomes
`scripts/build_vector_db.py`:

1. Excluding the Executive Summary / Project Navigator / System Architecture
   block (already served statically from `context/orientation/`) by heading
   name, not a hardcoded line range — survives future README edits.
2. `chunk_by_title`'s output chunks do NOT share an `id` with any source
   element (confirmed: 0/73 matched) — the doc's own flagged risk, real.
   Fixed here using `chunk.metadata.orig_elements[0].id` instead, which DOES
   match (confirmed: 73/73) — a direct reference to the real first element
   in that chunk, not a guess by ordinal position.
3. `el.id`, `el.category`, `el.metadata.category_depth` all confirmed present
   on this version.

In [1]:
import hashlib
import pathlib

from pydantic import BaseModel
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.html import partition_html


class Chunk(BaseModel):
    chunk_id: str
    doc_source: str    # which document this came from, e.g. "readme"
    file_path: str
    section: str | None
    length: int         # len(chunk_text), in characters
    chunk_text: str

In [2]:
path = pathlib.Path("../context/docs/index.html")
elements = partition_html(filename=str(path))
print(f"{len(elements)} elements parsed from {path}")

499 elements parsed from ..\context\docs\index.html


## Strip the orientation-bundle block by heading name

Skip everything from the "Executive Summary" depth-0 title (which contains
"Project Navigator" nested at depth 1) through "System Architecture", up to
— not including — the next depth-0 title, "Environment Setup & Provisioning".
Confirmed these are the real depth-0/1 boundaries against this file (not
assumed from a line-number guess).

In [3]:
EXCLUDE_START_TITLE = "Executive Summary"
EXCLUDE_END_TITLE = "Environment Setup & Provisioning"

filtered = []
skipping = False
for el in elements:
    is_top_title = el.category == "Title" and el.metadata.category_depth == 0
    if is_top_title and str(el) == EXCLUDE_START_TITLE:
        skipping = True
    if is_top_title and str(el) == EXCLUDE_END_TITLE:
        skipping = False
    if not skipping:
        filtered.append(el)

print(f"{len(elements)} -> {len(filtered)} elements after exclusion")
print(f"removed {len(elements) - len(filtered)} elements")

# Sanity check: none of the three excluded headings should survive.
remaining_titles = {str(el) for el in filtered if el.category == "Title"}
for bad in ("Executive Summary", "Project Navigator", "System Architecture"):
    assert bad not in remaining_titles, f"{bad!r} was not excluded"
print("confirmed: none of the 3 excluded headings remain")

499 -> 414 elements after exclusion
removed 85 elements
confirmed: none of the 3 excluded headings remain


## Build the breadcrumb stack (from the FILTERED elements, pre-chunking)

In [4]:
breadcrumbs: dict[str, str] = {}
stack: list[str] = []
for el in filtered:
    depth = el.metadata.category_depth or 0
    if el.category == "Title":
        stack[depth:] = [str(el)]  # push, truncating deeper levels
    breadcrumbs[el.id] = " > ".join(stack)

print(f"{len(breadcrumbs)} breadcrumb entries built")

414 breadcrumb entries built


## Chunk, then attach each chunk's breadcrumb via its first `orig_element`

In [5]:
def _chunk_id(file_path: str, text: str) -> str:
    return hashlib.sha256(f"{file_path}:{text}".encode()).hexdigest()[:16]


raw_chunks = chunk_by_title(filtered, max_characters=1500, overlap=150)
print(f"{len(raw_chunks)} chunks produced")

# path.as_posix(), not str(path): str() renders OS-native separators, so a
# backslash on Windows would fold into the hash and give a DIFFERENT
# chunk_id for identical content run from a different OS (e.g. Cloud Run
# later) -- breaking the "identical ids for unchanged content" guarantee
# _chunk_id exists for.
chunks: list[Chunk] = []
for c in raw_chunks:
    first_orig = c.metadata.orig_elements[0]
    crumb = breadcrumbs.get(first_orig.id, "")
    text = f"[{crumb}]\n{c}" if crumb else str(c)
    chunks.append(Chunk(
        chunk_id=_chunk_id(path.as_posix(), text),
        doc_source="readme",
        file_path=path.as_posix(),
        section=crumb or None,
        length=len(text),
        chunk_text=text,
    ))

missing_breadcrumb = sum(1 for c in chunks if c.section is None)
print(f"{missing_breadcrumb} / {len(chunks)} chunks have no breadcrumb")

66 chunks produced
0 / 66 chunks have no breadcrumb


In [6]:
from itertools import islice
for key, value in islice(breadcrumbs.items(), 10):
    print(f"{key} -> {value!r}")


07317e0b05e395ae7fcaf02a99f7e2da -> 'Environment Setup & Provisioning'
42e0f13f354b07175bc32917fa1c695c -> 'Environment Setup & Provisioning'
e7c9125114170fc64f4b3cf51389a6be -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
97cf11d22d4b45b811f15c417b2a8139 -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
418ab9fe7441c554399ff736301f0d10 -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
466def45b8b4dac7d0b827549da08ac5 -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
8c2ce71eafdbd605bd332e8d8fa500de -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
f0181bb7148c6f10ad5043a7fb007b8e -> 'Environment Setup & Provisioning > 1. Prerequisites & API Enablement'
e1aa01f7d33090686fffe5cae8e6c3d1 -> 'Environment Setup & Provisioning > 2. IAM Security & Permissions Configuration'
f3234a91eeef6156114574ba0eae47a7 -> 'Environment Setup & Provisioning > 2. IAM Security & Permissio

In [7]:
debug_stack: list[str] = []
for el in filtered[:15]:
    depth = el.metadata.category_depth or 0
    if el.category == "Title":
        debug_stack[depth:] = [str(el)]  # push, truncating deeper levels
    print(f"depth={depth:>2} category={el.category} stack={debug_stack}")


depth= 0 category=Title stack=['Environment Setup & Provisioning']
depth= 0 category=NarrativeText stack=['Environment Setup & Provisioning']
depth= 1 category=Title stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 0 category=NarrativeText stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 0 category=NarrativeText stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 0 category=UncategorizedText stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 0 category=CodeSnippet stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 0 category=NarrativeText stack=['Environment Setup & Provisioning', '1. Prerequisites & API Enablement']
depth= 1 category=Title stack=['Environment Setup & Provisioning', '2. IAM Security & Permissions Configuration']
depth= 0 category=NarrativeText stack=['Environment Setup & Provisioning', '2.

## Eyeball the results before this becomes build_vector_db.py

In [8]:
for c in chunks[:5]:
    print("=" * 80)
    print(c.chunk_text[:400])

[Environment Setup & Provisioning]
Environment Setup & Provisioning

This project is designed to run on the Google Cloud Platform (GCP), leveraging Google Cloud Shell, Cloud Storage (GCS), BigQuery, and Agent Platform Workbench with GPU acceleration. Follow the steps below to provision an identical workspace environment.

1. Prerequisites & API Enablement

GCP service APIs are specified in a versi
[Environment Setup & Provisioning > 2. IAM Security & Permissions Configuration]
2. IAM Security & Permissions Configuration

To execute native BigQuery operations, feature pipelines, and storage access without permission bottlenecks:

User Account: Ensure your GCP identity has the Owner, BigQuery Admin, and Service Usage Admin roles assigned.

Compute Engine Default Service Account: Grant the Com
[Environment Setup & Provisioning > 3. Data Ingestion: Kaggle to BigQuery Pipeline > Step 3.2: Download and Unpack Public Dataset]
Step 3.2: Download and Unpack Public Dataset

Bash (Cloud Shell)

k

In [9]:
lengths = [len(c.chunk_text) for c in chunks]
print(f"chunk count: {len(chunks)}")
print(f"chunk_text length -- min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths) / len(lengths):.0f}")

chunk count: 66
chunk_text length -- min: 231, max: 1610, avg: 1189
